In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [5]:
import numpy as np
import pandas as pd

np.random.seed(7)
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]

# Tabel referensi/master: target & manager per kategori (data ini relatif statis, jarang berubah)
data_produk = {
    "kategori": kategori_list,
    "target_bulanan": [50000000, 40000000, 30000000, 25000000, 20000000],
    "manager": ["Andi", "Budi", "Citra", "Dewi", "Eka"],
}
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))

# Tabel transaksi: data yang terus bertambah setiap hari
n = 300
data_transaksi = {
    "order_id": [f"O{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))

print("df_produk:")
df_produk.show()
print("df_transaksi (5 baris pertama dari total", df_transaksi.count(), "baris):")
df_transaksi.show(5)

df_produk:


+--------------------+--------------+-------+
|            kategori|target_bulanan|manager|
+--------------------+--------------+-------+
|          Elektronik|      50000000|   Andi|
|             Fashion|      40000000|   Budi|
|   Makanan & Minuman|      30000000|  Citra|
|Kesehatan & Kecan...|      25000000|   Dewi|
|        Rumah Tangga|      20000000|    Eka|
+--------------------+--------------+-------+



df_transaksi (5 baris pertama dari total 300 baris):
+--------+--------------------+--------+----------+
|order_id|            kategori|    kota|pendapatan|
+--------+--------------------+--------+----------+
|      O0|        Rumah Tangga|Magelang|    488643|
|      O1|             Fashion|Magelang|    401943|
|      O2|Kesehatan & Kecan...|    Solo|    452308|
|      O3|Kesehatan & Kecan...|Semarang|    421741|
|      O4|        Rumah Tangga|Semarang|    185244|
+--------+--------------------+--------+----------+
only showing top 5 rows



In [6]:
spark = SparkSession.builder.appName("Latihan5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

np.random.seed(7)
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))
df_transaksi.createOrReplaceTempView("transaksi")
df_produk.createOrReplaceTempView("produk")
print("Siap untuk latihan.")

Siap untuk latihan.


In [7]:
from pyspark.sql.functions import col

# Melakukan join antara df_transaksi dan df_produk, lalu filter kota "Solo"
hasil_soal_1 = df_transaksi.join(df_produk, on="kategori", how="inner") \
                           .filter(col("kota") == "Solo")

hasil_soal_1.show()

[Stage 7:============================================>              (3 + 1) / 4]

+----------+--------+----+----------+--------------+-------+
|  kategori|order_id|kota|pendapatan|target_bulanan|manager|
+----------+--------+----+----------+--------------+-------+
|Elektronik|     O10|Solo|     72294|      50000000|   Andi|
|Elektronik|     O12|Solo|     75566|      50000000|   Andi|
|Elektronik|     O14|Solo|    459719|      50000000|   Andi|
|Elektronik|     O21|Solo|    176776|      50000000|   Andi|
|Elektronik|     O22|Solo|     50323|      50000000|   Andi|
|Elektronik|     O24|Solo|    349635|      50000000|   Andi|
|Elektronik|     O32|Solo|    464460|      50000000|   Andi|
|Elektronik|     O44|Solo|    232991|      50000000|   Andi|
|Elektronik|     O46|Solo|    226277|      50000000|   Andi|
|Elektronik|     O70|Solo|    229714|      50000000|   Andi|
|Elektronik|    O103|Solo|    226138|      50000000|   Andi|
|Elektronik|    O107|Solo|    142577|      50000000|   Andi|
|Elektronik|    O118|Solo|    375622|      50000000|   Andi|
|Elektronik|    O130|Sol

In [9]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# Membuat window partisi berdasarkan kota, diurutkan berdasarkan pendapatan tertinggi (descending)
window_kota = Window.partitionBy("kota").orderBy(col("pendapatan").desc())

# Beri nomor urut menggunakan row_number(), ambil peringkat 1 saja
hasil_soal_2 = df_transaksi.withColumn("no_urut", row_number().over(window_kota)) \
                .filter(col("no_urut") == 1) \
                .drop("no_urut")

hasil_soal_2.show()

[Stage 11:>                                                         (0 + 4) / 4]

+--------+------------+--------+----------+
|order_id|    kategori|    kota|pendapatan|
+--------+------------+--------+----------+
|     O96|     Fashion|Magelang|    498770|
|    O147|Rumah Tangga|Semarang|    497370|
|    O286|Rumah Tangga|    Solo|    497826|
+--------+------------+--------+----------+



In [10]:
hasil_soal_3 = spark.sql("""
    SELECT kota, AVG(pendapatan) AS rata_rata_pendapatan
    FROM transaksi
    GROUP BY kota
    ORDER BY rata_rata_pendapatan DESC
""")

hasil_soal_3.show()

[Stage 14:>                                                         (0 + 4) / 4]

+--------+--------------------+
|    kota|rata_rata_pendapatan|
+--------+--------------------+
|    Solo|  278926.25925925927|
|Semarang|            267303.8|
|Magelang|   263705.6568627451|
+--------+--------------------+

